In [1]:
# === CELL 1 (Participant NB) — Who is this? ===
PARTICIPANT_NUM = int(input("Enter participant number (1-8): "))
print(f"Running study for Participant {PARTICIPANT_NUM}")

Enter participant number (1-8): 1
Running study for Participant 1


In [2]:
# === CELL 2 (Participant NB) — Load data ===
import numpy as np
import torch, torch.nn as nn, copy
from sklearn.datasets import load_breast_cancer
from sklearn.preprocessing import StandardScaler
from sklearn.model_selection import train_test_split

DATA_SEED, PARTITION_SEED, FAULT_RNG_SEED, MODEL_SEED = 42, 42, 123, 7
NUM_CLIENTS, FAULT_CLIENT, FAULT_ROUND = 8, 0, 8
WARMUP_ROUNDS, PERSISTENCE_ROUNDS, Z_THRESHOLD, NUM_ROUNDS = 5, 2, -1.0, 15

data = load_breast_cancer()
X, y = data.data, data.target
X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2, random_state=DATA_SEED, stratify=y)
scaler = StandardScaler().fit(X_train)
X_train, X_test = scaler.transform(X_train), scaler.transform(X_test)
print("Dataset loaded: breast cancer diagnostic data, simulating", NUM_CLIENTS, "hospital sites.")

Dataset loaded: breast cancer diagnostic data, simulating 8 hospital sites.


In [3]:
# === CELL 3 (Participant NB) — Partition into sites ===
def dirichlet_partition(y, num_clients, alpha, seed, min_size=15):
    rng = np.random.default_rng(seed)
    num_classes = len(np.unique(y))
    for attempt in range(50):
        class_idx = [np.where(y == c)[0].copy() for c in range(num_classes)]
        client_idx = [[] for _ in range(num_clients)]
        for c in range(num_classes):
            idx_c = class_idx[c]; rng.shuffle(idx_c)
            proportions = rng.dirichlet(alpha=[alpha]*num_clients)
            splits = (np.cumsum(proportions) * len(idx_c)).astype(int)[:-1]
            parts = np.split(idx_c, splits)
            for i, p in enumerate(parts): client_idx[i].extend(p.tolist())
        if min(len(x) for x in client_idx) >= min_size:
            return [np.array(idx) for idx in client_idx]
        seed += 1
    raise RuntimeError("fail")

client_indices = dirichlet_partition(y_train, NUM_CLIENTS, 0.5, PARTITION_SEED, min_size=15)
print(f"Partitioned into {NUM_CLIENTS} simulated sites with realistic uneven data distributions.")

Partitioned into 8 simulated sites with realistic uneven data distributions.


In [4]:
# === CELL 4 (Participant NB) — Model definition ===
class SmallMLP(nn.Module):
    def __init__(self, in_dim):
        super().__init__()
        self.net = nn.Sequential(nn.Linear(in_dim,32), nn.ReLU(), nn.Linear(32,16), nn.ReLU(), nn.Linear(16,2))
    def forward(self,x): return self.net(x)

def flat_params(sd): return torch.cat([v.float().view(-1) for v in sd.values()])

def local_train(global_state, X_c, y_c, in_dim, epochs=5, lr=0.05):
    model = SmallMLP(in_dim); model.load_state_dict(global_state)
    opt = torch.optim.SGD(model.parameters(), lr=lr); loss_fn = nn.CrossEntropyLoss()
    Xt = torch.tensor(X_c, dtype=torch.float32); yt = torch.tensor(y_c, dtype=torch.long)
    model.train()
    for _ in range(epochs):
        opt.zero_grad(); out = model(Xt); loss = loss_fn(out, yt); loss.backward(); opt.step()
    return model.state_dict(), loss.item(), len(y_c)

print("Model ready.")

Model ready.


In [5]:
# === CELL 5 (Participant NB) — Run FL simulation ===
torch.manual_seed(MODEL_SEED)
in_dim = X_train.shape[1]
global_model = SmallMLP(in_dim)
rng = np.random.default_rng(FAULT_RNG_SEED)
consecutive_low = [0]*NUM_CLIENTS
signal_history = []

for rnd in range(1, NUM_ROUNDS+1):
    global_state_before = copy.deepcopy(global_model.state_dict())
    global_flat_before = flat_params(global_state_before)
    client_states, client_weights = [], []
    for i, idx in enumerate(client_indices):
        Xc, yc = X_train[idx], y_train[idx].copy()
        if i == FAULT_CLIENT and rnd >= FAULT_ROUND:
            yc = rng.permutation(yc)
        state, loss, n = local_train(global_state_before, Xc, yc, in_dim)
        client_states.append(state); client_weights.append(n)

    deltas = [flat_params(s) - global_flat_before for s in client_states]
    mean_delta = torch.stack(deltas).mean(dim=0)
    sims = [torch.nn.functional.cosine_similarity(mean_delta.unsqueeze(0), d.unsqueeze(0)).item() for d in deltas]
    sims_arr = np.array(sims); mu, sigma = sims_arr.mean(), sims_arr.std()+1e-8
    z_scores = ((sims_arr - mu)/sigma).tolist()

    excluded = [False]*NUM_CLIENTS
    if rnd > WARMUP_ROUNDS:
        for i, z in enumerate(z_scores):
            consecutive_low[i] = consecutive_low[i]+1 if z < Z_THRESHOLD else 0
            if consecutive_low[i] >= PERSISTENCE_ROUNDS:
                excluded[i] = True

    adj_w = [w*0.05 if e else w for w,e in zip(client_weights, excluded)]
    total = sum(adj_w)
    avg_state = copy.deepcopy(client_states[0])
    for key in avg_state:
        avg_state[key] = sum(client_states[i][key].float()*(adj_w[i]/total) for i in range(NUM_CLIENTS))
    global_model.load_state_dict(avg_state)

    global_model.eval()
    with torch.no_grad():
        preds = global_model(torch.tensor(X_test, dtype=torch.float32)).argmax(dim=1).numpy()
    acc = (preds == y_test).mean()

    signal_history.append({"round": rnd, "test_accuracy": float(acc), "z_score": z_scores,
                            "cos_sim": sims, "excluded": excluded,
                            "effective_share": [a/total for a in adj_w]})

print(f"Simulation complete: {len(signal_history)} training rounds run across {NUM_CLIENTS} sites.")

Simulation complete: 15 training rounds run across 8 sites.


In [18]:
# === CELL 6 (Participant NB) — Explanation utilities [UPDATED: raw view now includes status flags] ===
def get_consecutive_excluded_count(history, round_idx, client_id):
    count = 0
    for i in range(round_idx, -1, -1):
        if history[i]["excluded"][client_id]: count += 1
        else: break
    return count

def get_notable_clients(record, z_threshold=-1.0, warmup_rounds=5):
    if record["round"] <= warmup_rounds: return []
    return [i for i, z in enumerate(record["z_score"]) if z < z_threshold or record["excluded"][i]]

def explain_contribution(record):
    shares = record["effective_share"]
    max_i, min_i = shares.index(max(shares)), shares.index(min(shares))
    if max(shares) - min(shares) < 0.05:
        return "All sites contributed roughly equally to the global model this round."
    return (f"Site {max_i} contributed the most to this round's model update "
            f"({shares[max_i]*100:.0f}% effective weight), while Site {min_i} "
            f"contributed the least ({shares[min_i]*100:.0f}%).")

def explain_divergence(record, client_id, excluded_this_round):
    z = record["z_score"][client_id]
    if excluded_this_round:
        return (f"Site {client_id}'s update looked substantially different from the rest of "
                f"the federation this round (divergence score {z:.2f}, well outside the normal range).")
    return (f"Site {client_id}'s update has just started looking unusual compared to the rest of "
            f"the federation this round (divergence score {z:.2f}). Not yet acted on - "
            f"the system waits for a consistent pattern before down-weighting a site.")

def explain_exclusion(history, round_idx, client_id):
    record = history[round_idx]
    if not record["excluded"][client_id]: return None
    streak = get_consecutive_excluded_count(history, round_idx, client_id)
    return (f"Site {client_id}'s contribution was significantly down-weighted this round. "
            f"This site's updates have looked unusual for {streak} consecutive round(s) - "
            f"this may indicate a data quality or labeling issue at this site, and is worth investigating "
            f"before trusting its future contributions.")

def generate_round_briefing(history, round_idx, z_threshold=-1.0, warmup_rounds=5):
    record = history[round_idx]
    lines = [f"--- Round {record['round']} briefing ---", f"Global test accuracy: {record['test_accuracy']*100:.1f}%",
             explain_contribution(record)]
    if record["round"] <= warmup_rounds:
        lines.append("Federation still stabilizing (warmup period) - anomaly detection not yet active.")
    else:
        notable = get_notable_clients(record, z_threshold, warmup_rounds)
        if not notable:
            lines.append("No sites showed unusual update behavior this round.")
        else:
            for cid in notable:
                lines.append(explain_divergence(record, cid, record["excluded"][cid]))
                excl = explain_exclusion(history, round_idx, cid)
                if excl: lines.append(excl)
    return "\n".join(lines)

def build_raw_view(record, z_threshold=-1.0, warmup_rounds=5):
    """Raw-numbers baseline with realistic status flags (mirrors existing FL
    monitoring dashboards) - still no interpretation, cause, or recommendation."""
    lines = [f"Round {record['round']} - Raw Signal Data", "-"*46,
             f"Global test accuracy: {record['test_accuracy']:.4f}", ""]
    lines.append(f"{'Site':<6}{'CosSim':>9}{'Z-score':>9}{'EffShare':>10}{'Excluded':>10}  {'Flag'}")
    in_warmup = record["round"] <= warmup_rounds
    for i in range(len(record["z_score"])):
        flag = ""
        if not in_warmup:
            if record["excluded"][i]: flag = "EXCLUDED"
            elif record["z_score"][i] < z_threshold: flag = "! unusual"
        lines.append(f"{i:<6}{record['cos_sim'][i]:>9.3f}{record['z_score'][i]:>9.2f}"
                      f"{record['effective_share'][i]:>10.3f}{str(record['excluded'][i]):>10}  {flag}")
    if in_warmup:
        lines.append(""); lines.append("(Federation still stabilizing - flags not yet active during warmup)")
    return "\n".join(lines)

print("Explanation utilities ready (raw view now flag-enhanced).")

Explanation utilities ready (raw view now flag-enhanced).


In [19]:
# === CELL 7 (Participant NB) — Condition assignment ===
SCENARIOS = [
    {"id": "S1_quiet_warmup", "round": 3},
    {"id": "S2_early_signal", "round": 8},
    {"id": "S3_fresh_exclusion", "round": 9},
    {"id": "S4_sustained_exclusion", "round": 15},
]
ids = [s["id"] for s in SCENARIOS]
FIXED_SPLITS = [
    {"raw": [ids[0], ids[1]], "briefing": [ids[2], ids[3]]},
    {"raw": [ids[2], ids[3]], "briefing": [ids[0], ids[1]]},
    {"raw": [ids[0], ids[2]], "briefing": [ids[1], ids[3]]},
    {"raw": [ids[1], ids[3]], "briefing": [ids[0], ids[2]]},
    {"raw": [ids[0], ids[3]], "briefing": [ids[1], ids[2]]},
    {"raw": [ids[1], ids[2]], "briefing": [ids[0], ids[3]]},
]

split = FIXED_SPLITS[(PARTICIPANT_NUM - 1) % len(FIXED_SPLITS)]
order = "raw_first" if PARTICIPANT_NUM % 2 == 1 else "briefing_first"

item_list = [{"id": sid, "cond": "raw_numbers"} for sid in split["raw"]] + \
            [{"id": sid, "cond": "briefing"} for sid in split["briefing"]]
if order == "briefing_first":
    item_list = item_list[2:] + item_list[:2]

scenario_by_id = {s["id"]: s["round"] for s in SCENARIOS}
print(f"Participant {PARTICIPANT_NUM}'s scenario order:")
for it in item_list:
    print(f"  {it['id']} -> {it['cond']}")

Participant 1's scenario order:
  S1_quiet_warmup -> raw_numbers
  S2_early_signal -> raw_numbers
  S3_fresh_exclusion -> briefing
  S4_sustained_exclusion -> briefing


In [20]:
# === CELL 8 (Participant NB) — Glossary ===
print("""
GLOSSARY (read this aloud once - applies to every scenario shown):

- Cosine similarity: how similar a site's update is to the group's average update
  this round. Closer to 1 = very similar; near 0 or negative = quite different.
- Z-score: how unusual a site's similarity is vs its peers this round, in std devs.
  Below about -1 is starting to look unusual; below -2 is quite unusual.
- Effective share: how much a site's update actually counted toward this round's
  result, after any down-weighting.
- Excluded: whether the system significantly reduced this site's influence this
  round due to persistent unusual behavior.
""")


GLOSSARY (read this aloud once - applies to every scenario shown):

- Cosine similarity: how similar a site's update is to the group's average update
  this round. Closer to 1 = very similar; near 0 or negative = quite different.
- Z-score: how unusual a site's similarity is vs its peers this round, in std devs.
  Below about -1 is starting to look unusual; below -2 is quite unusual.
- Effective share: how much a site's update actually counted toward this round's
  result, after any down-weighting.
- Excluded: whether the system significantly reduced this site's influence this
  round due to persistent unusual behavior.



In [22]:
# === CELL 9 (Participant NB) — Run this cell 4 times, once per scenario [UPDATED: briefing now includes raw numbers for verification] ===
SCENARIO_INDEX = 3  # <-- change to 0,1,2,3 and re-run this cell for each of the 4 scenarios

item = item_list[SCENARIO_INDEX]
round_num = scenario_by_id[item["id"]]
record = signal_history[round_num - 1]

print(f"=== Scenario {SCENARIO_INDEX+1} of 4: {item['id']} (Round {round_num}) — condition: {item['cond']} ===\n")

if item["cond"] == "raw_numbers":
    print(build_raw_view(record))
else:
    # briefing condition now shows raw numbers first, then the interpretation -
    # lets the participant verify the explanation against the underlying evidence
    print(build_raw_view(record))
    print("\n--- Plain-language interpretation of the above ---\n")
    print(generate_round_briefing(signal_history, round_num - 1))

print("\n--- Ask these live, type answers below ---")
q1 = input("Q1: In your own words, what's happening this round? ")
q2 = input("Q2: Is any site currently being down-weighted? (Yes/No/Not sure) ")
q3 = input("Q3: Trust in this round's model, 1-7, and why? ")
q4 = input("Q4: What, if anything, would you do next? ")
q5 = input("Q5: Confidence in that decision, 1-7? ")

if "responses" not in dir():
    responses = []
responses.append({"participant": PARTICIPANT_NUM, "scenario": item["id"], "round": round_num,
                   "condition": item["cond"], "q1": q1, "q2": q2, "q3": q3, "q4": q4, "q5": q5})
print(f"\nLogged. {len(responses)}/4 scenarios recorded so far.")

=== Scenario 4 of 4: S4_sustained_exclusion (Round 15) — condition: briefing ===

Round 15 - Raw Signal Data
----------------------------------------------
Global test accuracy: 0.9474

Site     CosSim  Z-score  EffShare  Excluded  Flag
0        -0.484    -2.25     0.004      True  EXCLUDED
1         0.267    -0.29     0.109     False  
2         0.206    -0.45     0.261     False  
3         0.623     0.64     0.045     False  
4         0.304    -0.19     0.111     False  
5         0.706     0.86     0.287     False  
6         0.751     0.98     0.062     False  
7         0.650     0.71     0.121     False  

--- Plain-language interpretation of the above ---

--- Round 15 briefing ---
Global test accuracy: 94.7%
Site 5 contributed the most to this round's model update (29% effective weight), while Site 0 contributed the least (0%).
Site 0's update looked substantially different from the rest of the federation this round (divergence score -2.25, well outside the normal range).
Sit

KeyboardInterrupt: Interrupted by user

In [13]:
# === CELL 10 (Participant NB) — Export results ===
import json
from google.colab import files

assert len(responses) == 4, f"Expected 4 responses, got {len(responses)} - did you run Cell 9 for all 4 scenarios?"

filename = f"participant_{PARTICIPANT_NUM:02d}_responses.json"
with open(filename, "w") as f:
    json.dump(responses, f, indent=2)

print(f"Saved {filename}. Downloading now...")
files.download(filename)

Saved participant_01_responses.json. Downloading now...


<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>